In [3]:
import pandas as pd
import numpy as np
from tqdm import tqdm
from methyldl.data.pseudo_bulk_generation import generate_pseudo_bulk
from methyldl.deconvolution.uxm import load_atlas
import pickle
import json
import os

In [4]:
methylbert_soft_labels_data_path = "/home/luna.kuleuven.be/u0169940/Repos/methyldl/Experiments/pseudobulk/methylBert_Loyfer_205files_U25_attentionClassifier_hg38_dmr_ctype_label_mincpg_4_minlen_10_postfiltered_min_length_50_soft_labels_pooled_jakkard_no_data_leak_d041/pseudobulk/pure_profiles.pkl"
methylbert_hard_labels_data_path = "/home/luna.kuleuven.be/u0169940/Repos/methyldl/Experiments/pseudobulk/methylBert_Loyfer_205files_U25_attentionClassifier_hg38_dmr_ctype_label_mincpg_4_minlen_10_postfiltered_min_length_50_hard_labels_minibatch_balanced/pseudobulk/pure_profiles.pkl"

In [6]:
with open(methylbert_soft_labels_data_path, "rb") as f:
    soft_profiles = pickle.load(f)

In [7]:
with open(methylbert_hard_labels_data_path, "rb") as f:
    hard_profiles = pickle.load(f)

In [15]:
atlas_location = "../../UXM_deconv/supplemental/Atlas.U25.l4.hg38.full.tsv"
atlas, ref_cells = load_atlas(atlas_location)

with open("../App/labels_dict.json", "rb") as f:
    labels_dict = json.load(f) 
labels_dict = {int(key):value for (key,value) in labels_dict.items()}
labels_dict_reversed = {y: x for (x, y) in labels_dict.items()}

In [19]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

def compare_model_predictions(data1, data2, true_props_array=None, class_name_key='dmr_ctype', class_idx_key='dmr_ctype_label', save_path=None, model_names = ["Model1", "Model2"]):
    """
    Compares prediction matrices from two models with potentially differing number of classes.
    Aligns both to a master union of classes, padding missing entries with NaNs.
    """
    # 1. Convert to DataFrames
    df1 = pd.DataFrame(data1)
    df2 = pd.DataFrame(data2)
    
    # 2. Build the master list of classes (Union of both datasets)
    all_labels = pd.concat([df1[[class_idx_key, class_name_key]], 
                            df2[[class_idx_key, class_name_key]]]).drop_duplicates()
    all_labels = all_labels.sort_values(class_idx_key).reset_index(drop=True)
    
    master_indices = all_labels[class_idx_key].tolist()
    master_names = all_labels[class_name_key].tolist()
    master_pred_cols = [f'prediction_{idx}_wavg' for idx in master_indices]
    
    # 3. Align DataFrames to the master list
    def align_dataframe(df, master_idx, master_cols):
        # Set index to class labels and reindex to master list (fills missing rows with NaN)
        aligned = df.set_index(class_idx_key).reindex(master_idx)
        # Ensure all master prediction columns exist (fills missing columns with NaN)
        for col in master_cols:
            if col not in aligned.columns:
                aligned[col] = np.nan
        # Reorder columns to match master list strictly
        return aligned[master_cols].values

    mat1 = align_dataframe(df1, master_indices, master_pred_cols)
    mat2 = align_dataframe(df2, master_indices, master_pred_cols)
    
    # Calculate difference (NaNs will propagate, leaving blank spots where classes are missing)
    diff_mat = mat2 - mat1
    
    # 4. Extract diagonals cleanly from the aligned square matrices
    diag1 = np.diag(mat1)
    diag2 = np.diag(mat2)

    # 6. Visualization
    fig = plt.figure(figsize=(20, 14))
    sns.set_theme(style="whitegrid")
    
    # Heatmap kwargs to handle NaNs cleanly
    heatmap_kwargs = {'cmap': 'viridis', 'yticklabels': master_names, 'xticklabels': master_names, 'mask': np.isnan(mat1)}
    
    ax1 = fig.add_subplot(2, 2, 1)
    sns.heatmap(mat1, ax=ax1, **heatmap_kwargs)
    ax1.set_title(f'{model_names[0]}', fontsize=14, fontweight='bold')
    ax1.set_ylabel('True Class')
    
    heatmap_kwargs['mask'] = np.isnan(mat2)
    ax2 = fig.add_subplot(2, 2, 2)
    sns.heatmap(mat2, ax=ax2, **heatmap_kwargs)
    ax2.set_title(f'{model_names[1]}', fontsize=14, fontweight='bold')
    
    ax3 = fig.add_subplot(2, 2, 3)
    vmax = np.nanmax(np.abs(diff_mat)) # Use nanmax to ignore NaNs
    sns.heatmap(diff_mat, cmap='coolwarm', ax=ax3, vmin=-vmax, vmax=vmax, 
                yticklabels=master_names, xticklabels=master_names, mask=np.isnan(diff_mat))
    ax3.set_title('Difference (Model 2 - Model 1)', fontsize=14, fontweight='bold')
    ax3.set_ylabel('True Class')
    
    # Diagonal Plot
    ax4 = fig.add_subplot(2, 2, 4)
    x = np.arange(len(master_names))
    width = 0.35
    
    ax4.bar(x - width/2, diag1, width, label=model_names[0], color='#440154', alpha=0.8)
    ax4.bar(x + width/2, diag2, width, label=model_names[1], color='#21918c', alpha=0.8)
    
    if true_props_array is not None:
        ax4.plot(x, true_props_array, color='red', marker='D', markersize=8, 
                 linestyle='None', zorder=3, label='Ground Truth Proportion')
                 
    ax4.set_xticks(x)
    ax4.set_xticklabels(master_names, rotation=90)
    ax4.set_title('Diagonal Scores vs. Ground Truth', fontsize=14, fontweight='bold')
    ax4.legend()

    plt.tight_layout()

    
    # 7. Build Analytical DataFrame
    df_dict = {
        'dmr_ctype_label': master_indices,
        'dmr_ctype': master_names,
        'Model_1_Target_Score': diag1,
        'Model_2_Target_Score': diag2,
        'Difference_(M2-M1)': diag2 - diag1
    }
    
    if true_props_array is not None:
        df_dict['True_Proportion'] = true_props_array
        df_dict['Model_1_Error'] = diag1 - true_props_array
        df_dict['Model_2_Error'] = diag2 - true_props_array
        
    diag_comparison_df = pd.DataFrame(df_dict)
    if save_path:
        # Ensure directory exists
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        plt.savefig(save_path, bbox_inches='tight', dpi=300)
        plt.close(fig) # CRITICAL: Releases memory
        return diff_mat, diag_comparison_df
    else:
        plt.show()
        return diff_mat, diag_comparison_df


In [74]:
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
def compare_model_predictions_v2(
    data1,
    data2,
    true_props_array=None,
    class_name_key='dmr_ctype',
    class_idx_key='dmr_ctype_label',
    save_path=None,
    model_names=("Model 1", "Model 2"),
    figsize=(18, 8),
):
    """
    Compare prediction matrices from two models in a single horizontal row of four panels:
        [Model1 heatmap] [Model2 heatmap] [Difference heatmap] [Horizontal bar chart]
 
    All panels share the y-axis (true class labels shown only on the leftmost panel),
    so each row reads as a single story across the figure.
 
    Classes present in either dataset are unioned; missing rows/columns become NaN
    and are masked in the heatmaps.
    """
    # 1. Convert to DataFrames
    df1 = pd.DataFrame(data1)
    df2 = pd.DataFrame(data2)
 
    # 2. Build the master list of classes (union of both datasets)
    all_labels = pd.concat([
        df1[[class_idx_key, class_name_key]],
        df2[[class_idx_key, class_name_key]],
    ]).drop_duplicates()
    all_labels = all_labels.sort_values(class_idx_key).reset_index(drop=True)
 
    master_indices = all_labels[class_idx_key].tolist()
    master_names = all_labels[class_name_key].tolist()
    master_pred_cols = [f'prediction_{idx}_wavg' for idx in master_indices]
 
    # 3. Align DataFrames to the master list
    def align_dataframe(df, master_idx, master_cols):
        aligned = df.set_index(class_idx_key).reindex(master_idx)
        for col in master_cols:
            if col not in aligned.columns:
                aligned[col] = np.nan
        return aligned[master_cols].values
 
    mat1 = align_dataframe(df1, master_indices, master_pred_cols)
    mat2 = align_dataframe(df2, master_indices, master_pred_cols)
    diff_mat = mat2 - mat1
 
    diag1 = np.diag(mat1)
    diag2 = np.diag(mat2)
 
    # 4. Figure layout: 4 panels in a row, shared y-axis.
    # Use add_axes-style manual placement so colorbars don't steal space from panels.
    sns.set_theme(style="whitegrid")
    fig = plt.figure(figsize=figsize)
 
    # Manual geometry: leave room on the right for two stacked colorbars
    left, right = 0.08, 0.90
    bottom, top = 0.22, 0.92
    total_w = right - left
    wspace = 0.015
    ratios = np.array([1.0, 1.0, 1.0, 0.75])
    widths = (total_w - 3 * wspace) * ratios / ratios.sum()
    lefts = [left]
    for w in widths[:-1]:
        lefts.append(lefts[-1] + w + wspace)
    height = top - bottom
 
    ax1 = fig.add_axes([lefts[0], bottom, widths[0], height])
    ax2 = fig.add_axes([lefts[1], bottom, widths[1], height], sharey=ax1)
    ax3 = fig.add_axes([lefts[2], bottom, widths[2], height], sharey=ax1)
    ax4 = fig.add_axes([lefts[3], bottom, widths[3], height], sharey=ax1)
 
    # Colorbar axes stacked on the right, outside the panels
    cbar_w = 0.012
    cbar_gap = 0.015
    cbar_x = right + cbar_gap
    cax_pred = fig.add_axes([cbar_x, bottom + height * 0.52, cbar_w, height * 0.40])
    cax_diff = fig.add_axes([cbar_x, bottom + height * 0.06, cbar_w, height * 0.40])
 
    # Shared scale for the two prediction heatmaps
    vmax_pred = np.nanmax([np.nanmax(mat1), np.nanmax(mat2)])
    vmin_pred = 0.0
    pred_cmap = 'viridis'
 
    # --- Panel 1: Model 1 heatmap (with y-axis labels) ---
    sns.heatmap(
        mat1, ax=ax1, cmap=pred_cmap, vmin=vmin_pred, vmax=vmax_pred,
        cbar=False, mask=np.isnan(mat1),
        xticklabels=master_names, yticklabels=master_names,
    )
    ax1.set_title(model_names[0], fontsize=11, fontweight='bold')
    ax1.set_ylabel('True class', fontsize=10)
    ax1.set_xlabel('Predicted class', fontsize=9)
    ax1.tick_params(axis='x', rotation=90, labelsize=7)
    ax1.tick_params(axis='y', labelsize=7)
 
    # --- Panel 2: Model 2 heatmap (no y-labels) ---
    sns.heatmap(
        mat2, ax=ax2, cmap=pred_cmap, vmin=vmin_pred, vmax=vmax_pred,
        cbar=False, mask=np.isnan(mat2),
        xticklabels=master_names, yticklabels=False,
    )
    ax2.set_title(model_names[1], fontsize=11, fontweight='bold')
    ax2.set_xlabel('Predicted class', fontsize=9)
    ax2.tick_params(axis='x', rotation=90, labelsize=7)
 
    # --- Panel 3: Difference heatmap ---
    vmax_diff = np.nanmax(np.abs(diff_mat)) if np.any(~np.isnan(diff_mat)) else 1.0
    sns.heatmap(
        diff_mat, ax=ax3, cmap='coolwarm', vmin=-vmax_diff, vmax=vmax_diff,
        cbar=False, mask=np.isnan(diff_mat),
        xticklabels=master_names, yticklabels=False,
    )
    ax3.set_title('Difference', fontsize=11, fontweight='bold')
    ax3.set_xlabel('Predicted class', fontsize=9)
    ax3.tick_params(axis='x', rotation=90, labelsize=7)
 
    # --- Colorbars (manually placed) ---
    pred_norm = Normalize(vmin=vmin_pred, vmax=vmax_pred)
    pred_sm = ScalarMappable(norm=pred_norm, cmap=pred_cmap)
    pred_sm.set_array([])
    cbar_pred = fig.colorbar(pred_sm, cax=cax_pred)
    cbar_pred.ax.tick_params(labelsize=7)
    cbar_pred.set_label('Prediction', fontsize=8)
 
    diff_norm = Normalize(vmin=-vmax_diff, vmax=vmax_diff)
    diff_sm = ScalarMappable(norm=diff_norm, cmap='RdBu_r')
    diff_sm.set_array([])
    cbar_diff = fig.colorbar(diff_sm, cax=cax_diff)
    cbar_diff.ax.tick_params(labelsize=7)
    cbar_diff.set_label(f'{model_names[1]} − {model_names[0]}', fontsize=8)
 
    # --- Panel 4: Horizontal bar chart of diagonal scores ---
    y = np.arange(len(master_names))
    bar_h = 0.38
 
    ax4.barh(y - bar_h / 2, diag1, bar_h,
             label=model_names[0], color='#440154', alpha=0.85)
    ax4.barh(y + bar_h / 2, diag2, bar_h,
             label=model_names[1], color='#21918c', alpha=0.85)
 
    if true_props_array is not None:
        ax4.plot(
            true_props_array, y,
            color='red', marker='D', markersize=5,
            linestyle='None', zorder=3, label='Ground truth',
        )
 
    # sharey with ax1 means ax4 inherits heatmap row ordering (row 0 at top);
    # but heatmap sets its own ylim, so we don't need to flip manually.
    ax4.tick_params(axis='y', labelleft=False)
    ax4.set_xlabel('Diagonal score', fontsize=9)
    ax4.set_title('Diagonal vs. ground truth', fontsize=11, fontweight='bold')
    xmax = max(1.0, np.nanmax([np.nanmax(diag1), np.nanmax(diag2)]) * 1.05)
    if true_props_array is not None:
        xmax = max(xmax, np.nanmax(true_props_array) * 1.05)
    ax4.set_xlim(0, xmax)
    ax4.tick_params(axis='x', labelsize=7)
    ax4.grid(axis='x', alpha=0.4)
    ax4.grid(axis='y', visible=False)
    ax4.legend(loc='lower right', fontsize=7, framealpha=0.9)
 
    # 5. Analytical DataFrame
    df_dict = {
        'dmr_ctype_label': master_indices,
        'dmr_ctype': master_names,
        f'{model_names[0]}_target_score': diag1,
        f'{model_names[1]}_target_score': diag2,
        f'difference_({model_names[1]}-{model_names[0]})': diag2 - diag1,
    }
    if true_props_array is not None:
        df_dict['true_proportion'] = true_props_array
        df_dict[f'{model_names[0]}_error'] = diag1 - true_props_array
        df_dict[f'{model_names[1]}_error'] = diag2 - true_props_array
    diag_comparison_df = pd.DataFrame(df_dict)
 
    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True) if os.path.dirname(save_path) else None
        plt.savefig(save_path, bbox_inches='tight', dpi=300)
        plt.close(fig)
    else:
        plt.show()
 
    return diff_mat, diag_comparison_df

In [78]:
from matplotlib.patches import Rectangle

def compare_model_predictions_v3(
    data1,
    data2,
    true_props_array=None,
    class_name_key='dmr_ctype',
    class_idx_key='dmr_ctype_label',
    save_path=None,
    model_names=("Model 1", "Model 2"),
    figsize=(15, 6),
    highlight_color='red',
    highlight_lw=1.8,
    fontsize = 12
):
    """
    Compare prediction matrices from two models as three heatmaps in a row:
        [Model 1]  [Model 2]  [Difference (M2 - M1)]

    All panels share the y-axis. Ticks and colorbars are suppressed for a clean
    paper figure. Common axis labels are drawn once at the figure level.

    If `true_props_array` is provided, the row and column of the class with the
    largest ground-truth value are highlighted with a colored rectangle in every
    panel (so the reader can trace the true class across all three matrices).

    Classes present in either dataset are unioned; missing rows/columns become
    NaN and are masked in the heatmaps.
    """
    # 1. Convert to DataFrames
    df1 = pd.DataFrame(data1)
    df2 = pd.DataFrame(data2)

    # 2. Build the master list of classes (union of both datasets)
    all_labels = pd.concat([
        df1[[class_idx_key, class_name_key]],
        df2[[class_idx_key, class_name_key]],
    ]).drop_duplicates()
    all_labels = all_labels.sort_values(class_idx_key).reset_index(drop=True)

    master_indices = all_labels[class_idx_key].tolist()
    master_names = all_labels[class_name_key].tolist()
    master_pred_cols = [f'prediction_{idx}_wavg' for idx in master_indices]
    n = len(master_names)

    # 3. Align DataFrames to the master list
    def align_dataframe(df, master_idx, master_cols):
        aligned = df.set_index(class_idx_key).reindex(master_idx)
        for col in master_cols:
            if col not in aligned.columns:
                aligned[col] = np.nan
        return aligned[master_cols].values

    mat1 = align_dataframe(df1, master_indices, master_pred_cols)
    mat2 = align_dataframe(df2, master_indices, master_pred_cols)
    diff_mat = mat2 - mat1

    diag1 = np.diag(mat1)
    diag2 = np.diag(mat2)

    # Determine the ground-truth row/column to highlight
    gt_idx = None
    if true_props_array is not None:
        true_props_array = np.asarray(true_props_array, dtype=float)
        if np.any(~np.isnan(true_props_array)):
            gt_idx = int(np.nanargmax(true_props_array))

    def draw_highlight(ax, idx, n_cells):
        """Outline the idx-th row and idx-th column on a heatmap axis."""
        if idx is None:
            return
        # heatmap cell coordinates: column i spans x=[i, i+1], row j spans y=[j, j+1]
        # Row rectangle (full width, one row tall)
        ax.add_patch(Rectangle(
            (0, idx), n_cells, 1,
            fill=False, edgecolor=highlight_color, linewidth=highlight_lw,
            zorder=5, clip_on=False,
        ))
        # Column rectangle (full height, one column wide)
        ax.add_patch(Rectangle(
            (idx, 0), 1, n_cells,
            fill=False, edgecolor=highlight_color, linewidth=highlight_lw,
            zorder=5, clip_on=False,
        ))

    # 4. Figure layout: 3 heatmap panels in a row, shared y-axis
    sns.set_theme(style="white")  # no gridlines behind heatmaps
    fig = plt.figure(figsize=figsize)

    # Manual geometry: leave room for common x/y labels in the margins
    left, right = 0.06, 0.985
    bottom, top = 0.08, 0.93
    total_w = right - left
    wspace = 0.02
    widths = [(total_w - 2 * wspace) / 3.0] * 3
    lefts = [left]
    for w in widths[:-1]:
        lefts.append(lefts[-1] + w + wspace)
    height = top - bottom

    ax1 = fig.add_axes([lefts[0], bottom, widths[0], height])
    ax2 = fig.add_axes([lefts[1], bottom, widths[1], height], sharey=ax1)
    ax3 = fig.add_axes([lefts[2], bottom, widths[2], height], sharey=ax1)

    # Shared scale for the two prediction heatmaps
    vmax_pred = np.nanmax([np.nanmax(mat1), np.nanmax(mat2)])
    vmin_pred = 0.0
    pred_cmap = 'viridis'

    # --- Panel 1: Model 1 heatmap ---
    sns.heatmap(
        mat1, ax=ax1, cmap=pred_cmap, vmin=vmin_pred, vmax=vmax_pred,
        cbar=False, mask=np.isnan(mat1),
        xticklabels=False, yticklabels=False,
    )
    ax1.set_title(model_names[0], fontsize=fontsize, fontweight='bold')
    ax1.tick_params(left=False, bottom=False)
    draw_highlight(ax1, gt_idx, n)

    # --- Panel 2: Model 2 heatmap ---
    sns.heatmap(
        mat2, ax=ax2, cmap=pred_cmap, vmin=vmin_pred, vmax=vmax_pred,
        cbar=False, mask=np.isnan(mat2),
        xticklabels=False, yticklabels=False,
    )
    ax2.set_title(model_names[1], fontsize=fontsize, fontweight='bold')
    ax2.tick_params(left=False, bottom=False)
    draw_highlight(ax2, gt_idx, n)

    # --- Panel 3: Difference heatmap ---
    vmax_diff = np.nanmax(np.abs(diff_mat)) if np.any(~np.isnan(diff_mat)) else 1.0
    # vmax_diff = np.nanpercentile(np.abs(diff_mat), 99.8)
    sns.heatmap(
        diff_mat, ax=ax3, cmap='RdBu_r', vmin=-vmax_diff, vmax=vmax_diff,
        cbar=False, mask=np.isnan(diff_mat),
        xticklabels=False, yticklabels=False,
    )
    ax3.set_title('Difference', fontsize=fontsize, fontweight='bold')
    ax3.tick_params(left=False, bottom=False)
    draw_highlight(ax3, gt_idx, n)

    # --- Common axis labels at the figure level ---
    fig.text(
        0.52, 0.025, 'Cell Type',
        ha='center', va='bottom', fontsize=fontsize, fontweight='bold',
    )
    fig.text(
        0.042, (bottom + top) / 2, 'DMR Cell Type Group',
        ha='left', va='center', rotation=90, fontsize=fontsize, fontweight='bold',
    )

    # 5. Analytical DataFrame
    df_dict = {
        'dmr_ctype_label': master_indices,
        'dmr_ctype': master_names,
        f'{model_names[0]}_target_score': diag1,
        f'{model_names[1]}_target_score': diag2,
        f'difference_({model_names[1]}-{model_names[0]})': diag2 - diag1,
    }
    if true_props_array is not None:
        df_dict['true_proportion'] = true_props_array
        df_dict[f'{model_names[0]}_error'] = diag1 - true_props_array
        df_dict[f'{model_names[1]}_error'] = diag2 - true_props_array
    diag_comparison_df = pd.DataFrame(df_dict)

    if save_path:
        if os.path.dirname(save_path):
            os.makedirs(os.path.dirname(save_path), exist_ok=True)
        plt.savefig(save_path, bbox_inches='tight', dpi=300)
        plt.close(fig)
    else:
        plt.show()

    return diff_mat, diag_comparison_df

In [81]:
def compare_model_predictions_v3(
    data1,
    data2,
    true_props_array=None,
    class_name_key='dmr_ctype',
    class_idx_key='dmr_ctype_label',
    save_path=None,
    model_names=("Model 1", "Model 2"),
    figsize=(16, 6),
    highlight_color='red',
    highlight_lw=1.8,
    fontsize=12,
    diff_cmap='RdBu_r',
):
    """
    Compare prediction matrices from two models as three heatmaps in a row:
        [Model 1]  [Model 2]  [Difference (M2 − M1)]
 
    Models may have different numbers of prediction columns (e.g. 39 vs 40).
    Each model's heatmap shows ALL of its own columns. The difference heatmap
    is computed only over the intersection of columns present in both models.
 
    Rows (DMR groups) are union-aligned: every group present in either model
    appears as a row; missing rows are NaN-masked.
 
    If `true_props_array` is provided, the row and column of the class with the
    largest ground-truth value are highlighted with a colored rectangle in every
    panel where that column exists.
    """
    # ── 1. Convert to DataFrames ──
    df1 = pd.DataFrame(data1)
    df2 = pd.DataFrame(data2)
 
    # ── 2. Master ROW index: union of row labels from both models ──
    all_labels = pd.concat([
        df1[[class_idx_key, class_name_key]],
        df2[[class_idx_key, class_name_key]],
    ]).drop_duplicates()
    all_labels = all_labels.sort_values(class_idx_key).reset_index(drop=True)
 
    master_row_indices = all_labels[class_idx_key].tolist()
    master_row_names = all_labels[class_name_key].tolist()
    n_rows = len(master_row_indices)
 
    # ── 3. Per-model COLUMN indices: each model keeps its own columns ──
    def get_pred_cols(df):
        """Return sorted (col_indices, col_names, col_strings) for one model."""
        raw_cols = [c for c in df.columns if c.startswith('prediction_') and c.endswith('_wavg')]
        # Sort by the numeric index, not lexicographically
        raw_cols_with_idx = [(int(c.split('_')[1]), c) for c in raw_cols]
        raw_cols_with_idx.sort(key=lambda x: x[0])
        col_indices = [idx for idx, _ in raw_cols_with_idx]
        pred_cols = [col for _, col in raw_cols_with_idx]
        # Build index→name map from the model's own label data
        idx_to_name = dict(zip(df[class_idx_key], df[class_name_key]))
        # For columns whose index isn't in the model's rows (e.g. background),
        # fall back to a generic name
        col_names = [idx_to_name.get(i, f'class_{i}') for i in col_indices]
        return col_indices, col_names, pred_cols
 
    col_idx1, col_names1, pred_cols1 = get_pred_cols(df1)
    col_idx2, col_names2, pred_cols2 = get_pred_cols(df2)
    n_cols1 = len(col_idx1)
    n_cols2 = len(col_idx2)
 
    # ── 4. Align each model's rows to the master row index ──
    def align_rows(df, master_idx, pred_cols):
        aligned = df.set_index(class_idx_key).reindex(master_idx)
        # Ensure all requested pred_cols exist (fills missing with NaN)
        for col in pred_cols:
            if col not in aligned.columns:
                aligned[col] = np.nan
        return aligned[pred_cols].values
 
    mat1 = align_rows(df1, master_row_indices, pred_cols1)  # (n_rows, n_cols1)
    mat2 = align_rows(df2, master_row_indices, pred_cols2)  # (n_rows, n_cols2)
 
    # ── 5. Difference over INTERSECTING columns only ──
    shared_col_idx = sorted(set(col_idx1) & set(col_idx2))
    shared_col_names = []
    # Column positions in each model's matrix for the shared subset
    pos_in_1 = []
    pos_in_2 = []
    for idx in shared_col_idx:
        pos_in_1.append(col_idx1.index(idx))
        pos_in_2.append(col_idx2.index(idx))
        # Use name from model 1 (they should agree for shared columns)
        shared_col_names.append(col_names1[col_idx1.index(idx)])
 
    n_shared = len(shared_col_idx)
    diff_mat = mat2[:, pos_in_2] - mat1[:, pos_in_1]  # (n_rows, n_shared)
 
    # ── 6. Diagonal scores (over shared columns only, matched by row index) ──
    diag1 = np.full(n_rows, np.nan)
    diag2 = np.full(n_rows, np.nan)
    for i, row_idx in enumerate(master_row_indices):
        if row_idx in col_idx1:
            j = col_idx1.index(row_idx)
            diag1[i] = mat1[i, j]
        if row_idx in col_idx2:
            j = col_idx2.index(row_idx)
            diag2[i] = mat2[i, j]
 
    # ── 7. Ground-truth highlight indices (per-panel) ──
    gt_row = None
    gt_col1 = None  # column index in mat1
    gt_col2 = None  # column index in mat2
    gt_col_diff = None  # column index in diff_mat
    if true_props_array is not None:
        true_props_array = np.asarray(true_props_array, dtype=float)
        if np.any(~np.isnan(true_props_array)):
            gt_row = int(np.nanargmax(true_props_array))
            gt_label = master_row_indices[gt_row]
            if gt_label in col_idx1:
                gt_col1 = col_idx1.index(gt_label)
            if gt_label in col_idx2:
                gt_col2 = col_idx2.index(gt_label)
            if gt_label in shared_col_idx:
                gt_col_diff = shared_col_idx.index(gt_label)
 
    def draw_highlight(ax, row_idx, col_idx, n_rows_, n_cols_):
        """Outline a row and/or column on a heatmap."""
        if row_idx is not None:
            ax.add_patch(Rectangle(
                (0, row_idx), n_cols_, 1,
                fill=False, edgecolor=highlight_color, linewidth=highlight_lw,
                zorder=5, clip_on=False,
            ))
        if col_idx is not None:
            ax.add_patch(Rectangle(
                (col_idx, 0), 1, n_rows_,
                fill=False, edgecolor=highlight_color, linewidth=highlight_lw,
                zorder=5, clip_on=False,
            ))
 
    # ── 8. Figure layout ──
    # Panel widths proportional to column counts so cells stay roughly square
    sns.set_theme(style="white")
    fig = plt.figure(figsize=figsize)
 
    left, right = 0.06, 0.985
    bottom, top = 0.08, 0.93
    total_w = right - left
    wspace = 0.02
    raw_ratios = np.array([n_cols1, n_cols2, n_shared], dtype=float)
    widths = (total_w - 2 * wspace) * raw_ratios / raw_ratios.sum()
    lefts = [left]
    for w in widths[:-1]:
        lefts.append(lefts[-1] + w + wspace)
    height = top - bottom
 
    ax1 = fig.add_axes([lefts[0], bottom, widths[0], height])
    ax2 = fig.add_axes([lefts[1], bottom, widths[1], height], sharey=ax1)
    ax3 = fig.add_axes([lefts[2], bottom, widths[2], height], sharey=ax1)
 
    # Shared color scale for the two prediction heatmaps
    vmax_pred = np.nanmax([np.nanmax(mat1), np.nanmax(mat2)])
    vmin_pred = 0.0
    pred_cmap = 'viridis'
 
    # --- Panel 1: Model 1 ---
    sns.heatmap(
        mat1, ax=ax1, cmap=pred_cmap, vmin=vmin_pred, vmax=vmax_pred,
        cbar=False, mask=np.isnan(mat1),
        xticklabels=False, yticklabels=False,
    )
    ax1.set_title(model_names[0], fontsize=fontsize, fontweight='bold')
    ax1.tick_params(left=False, bottom=False)
    draw_highlight(ax1, gt_row, gt_col1, n_rows, n_cols1)
 
    # --- Panel 2: Model 2 ---
    sns.heatmap(
        mat2, ax=ax2, cmap=pred_cmap, vmin=vmin_pred, vmax=vmax_pred,
        cbar=False, mask=np.isnan(mat2),
        xticklabels=False, yticklabels=False,
    )
    ax2.set_title(model_names[1], fontsize=fontsize, fontweight='bold')
    ax2.tick_params(left=False, bottom=False)
    draw_highlight(ax2, gt_row, gt_col2, n_rows, n_cols2)
 
    # --- Panel 3: Difference (intersection only) ---
    vmax_diff = np.nanmax(np.abs(diff_mat)) if np.any(~np.isnan(diff_mat)) else 1.0
    sns.heatmap(
        diff_mat, ax=ax3, cmap=diff_cmap, vmin=-vmax_diff, vmax=vmax_diff,
        cbar=False, mask=np.isnan(diff_mat),
        xticklabels=False, yticklabels=False,
    )
    ax3.set_title('Difference', fontsize=fontsize, fontweight='bold')
    ax3.tick_params(left=False, bottom=False)
    draw_highlight(ax3, gt_row, gt_col_diff, n_rows, n_shared)
 
    # --- Common axis labels ---
    fig.text(
        0.52, 0.025, 'Cell Type',
        ha='center', va='bottom', fontsize=fontsize, fontweight='bold',
    )
    fig.text(
        0.042, (bottom + top) / 2, 'DMR Cell Type Group',
        ha='left', va='center', rotation=90, fontsize=fontsize, fontweight='bold',
    )
 
    # ── 9. Analytical DataFrame ──
    df_dict = {
        'dmr_ctype_label': master_row_indices,
        'dmr_ctype': master_row_names,
        f'{model_names[0]}_target_score': diag1,
        f'{model_names[1]}_target_score': diag2,
        f'difference_({model_names[1]}-{model_names[0]})': diag2 - diag1,
    }
    if true_props_array is not None:
        df_dict['true_proportion'] = true_props_array
        df_dict[f'{model_names[0]}_error'] = diag1 - true_props_array
        df_dict[f'{model_names[1]}_error'] = diag2 - true_props_array
    diag_comparison_df = pd.DataFrame(df_dict)
 
    if save_path:
        if os.path.dirname(save_path):
            os.makedirs(os.path.dirname(save_path), exist_ok=True)
        plt.savefig(save_path, bbox_inches='tight', dpi=300)
        plt.close(fig)
    else:
        plt.show()
 
    return diff_mat, diag_comparison_df

In [82]:
for i in range(39):
    # Construct a unique filename for each iteration
    file_name = f"comparison_pure_{labels_dict[i]}.png"
    full_path = os.path.join("features_comparison_for_supplementary", file_name)
    
    print(f"Processing and saving iteration {i}...")
    
    # Call the modified function
    compare_model_predictions_v3(
        hard_profiles[i][1][0], 
        soft_profiles[i][1][0], 
        true_props_array=soft_profiles[i][0], # Assuming this is your dict/array
        save_path=full_path,
        model_names=["MethylBERT Hard Labels", "MethylBERT Soft Labels with Pooling"],
        figsize=(18,5),
        fontsize=18
    )

Processing and saving iteration 0...
Processing and saving iteration 1...
Processing and saving iteration 2...
Processing and saving iteration 3...
Processing and saving iteration 4...
Processing and saving iteration 5...
Processing and saving iteration 6...
Processing and saving iteration 7...
Processing and saving iteration 8...
Processing and saving iteration 9...
Processing and saving iteration 10...
Processing and saving iteration 11...
Processing and saving iteration 12...
Processing and saving iteration 13...
Processing and saving iteration 14...
Processing and saving iteration 15...
Processing and saving iteration 16...
Processing and saving iteration 17...
Processing and saving iteration 18...
Processing and saving iteration 19...
Processing and saving iteration 20...
Processing and saving iteration 21...
Processing and saving iteration 22...
Processing and saving iteration 23...
Processing and saving iteration 24...
Processing and saving iteration 25...
Processing and saving 